# 切分策略选择与评估

切分没有脱离数据集的统一最优参数。应先选择合理 baseline，再使用真实问题和标准答案评估，而不是只观察 Chunk 看起来是否整齐。

## 起点建议

| 数据类型 | 推荐起点 |
| --- | --- |
| 普通自然语言文本 | `RecursiveCharacterTextSplitter` |
| Markdown / HTML | 标题结构切分 + 递归长度兜底 |
| JSON | 按业务 Schema 构造 Document，或 `RecursiveJsonSplitter` |
| 代码 | `from_language()` baseline；高级场景使用 AST |
| 复杂 PDF / DOCX | Docling、MinerU 等结构解析 + Token 上限 |
| 缺少可靠结构、主题变化明显 | 评估语义分块是否值得额外成本 |

In [ ]:
from statistics import mean
from langchain_core.documents import Document

def describe_chunks(chunks: list[Document]) -> dict:
    lengths = [len(chunk.page_content) for chunk in chunks]
    return {
        "count": len(lengths),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "avg_chars": round(mean(lengths), 2),
        "with_source": sum("source" in chunk.metadata for chunk in chunks),
        "with_section": sum(
            bool({"chapter", "section", "subsection"} & chunk.metadata.keys())
            for chunk in chunks
        ),
    }

# 可传入前面 Notebook 生成的 chunks 或 final_chunks。
# print(describe_chunks(final_chunks))


## 真正需要评估的指标

- **Retrieval Recall@K**：答案证据是否出现在 Top-K Chunk 中。
- **Context Precision**：返回上下文中有多少内容真正相关。
- **边界完整率**：定义、步骤、表格行或代码函数是否被错误截断。
- **引用准确率**：是否能稳定回到 source、page、section 和位置。
- **索引规模**：Chunk 数量、Embedding Token 和存储体积。
- **延迟与成本**：构建索引、检索和生成阶段的耗时与费用。

推荐至少比较三组：固定/递归切分 baseline、结构切分、结构优先加长度兜底。只有评估结果才能说明复杂切分策略是否带来真实收益。

## 最终结论

多数项目可以从 `RecursiveCharacterTextSplitter` 起步；文档有可靠结构时，应优先保留结构；复杂 PDF 和多模态内容需要上游解析器；语义分块和多视图索引属于需要用评估证明价值的高级策略。